# California Housing - Classical ML Capstone

Dataset: `sklearn.datasets.fetch_california_housing()` - verified via scikit-learn's own docs (see README.md). 20,640 rows, 8 features, target = median house value in $100,000 units.

**Requires internet on first run** (downloads and caches the dataset). See README.md's network note.

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocess import load_dataset, make_income_bucket, stratified_split, FeatureEngineer

sns.set_theme(style="whitegrid")

## Milestone 1 - Validation setup (leakage-safe split)

In [ ]:
df = load_dataset()
print(df.shape)
df.head()

In [ ]:
strata = make_income_bucket(df)
print(strata.value_counts(normalize=True).sort_index())

In [ ]:
X_train, X_test, y_train, y_test = stratified_split(df)
print(f"train={len(X_train)}  test={len(X_test)}")
# Set aside X_test / y_test now. Do not touch them again until Milestone 6.

## Milestone 2 - Baseline: plain LinearRegression

In [ ]:
from src.train import evaluate_baseline
from sklearn.model_selection import train_test_split

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
baseline_rmse = evaluate_baseline(X_tr, y_tr, X_val, y_val)
print(f"Baseline val RMSE: {baseline_rmse:.4f}  (units: $100k)")

## Milestone 3 - Feature engineering

`FeatureEngineer` (in `src/preprocess.py`) adds `rooms_per_household`, `bedrooms_per_room`, `population_per_household`. Check their correlation with the target below before trusting they help.

In [ ]:
fe = FeatureEngineer()
X_train_fe = fe.fit_transform(X_train)
corr_with_target = X_train_fe.assign(target=y_train.values).corr()["target"].sort_values(ascending=False)
corr_with_target

**Written justification:** _(TODO(you) - which derived features actually correlate meaningfully with the target? Keep or drop accordingly, and say why.)_

## Milestone 4 - Model comparison (cross-validated search)

Uses `src/train.py::run_search`, which wraps `FeatureEngineer -> StandardScaler -> estimator` in a single `Pipeline`, so the search never leaks test-fold statistics into the feature engineering step.

In [ ]:
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from src.train import run_search

cv = KFold(n_splits=5, shuffle=True, random_state=42)

searches = [
    ("ridge", Ridge(), {"estimator__alpha": [0.1, 1.0, 10.0, 50.0]}),
    ("random_forest", RandomForestRegressor(random_state=42, n_jobs=-1),
     {"estimator__n_estimators": [200, 400], "estimator__max_depth": [None, 10, 20]}),
    ("hist_gb", HistGradientBoostingRegressor(random_state=42),
     {"estimator__max_depth": [None, 6, 10], "estimator__learning_rate": [0.05, 0.1]}),
]

results = []
for name, estimator, grid in searches:
    r = run_search(name, estimator, grid, X_train, y_train, cv)
    print(f"{name}: CV RMSE = {r['cv_rmse_mean']:.4f} +/- {r['cv_rmse_std']:.4f}  best_params={r['best_params']}")
    results.append(r)

In [ ]:
best = min(results, key=lambda r: r["cv_rmse_mean"])
print(f"Best model: {best['name']} (CV RMSE {best['cv_rmse_mean']:.4f})")
best_model = best["best_estimator"]

## Milestone 5 - Diagnostics: learning curve

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train, y_train, cv=5,
    scoring="neg_root_mean_squared_error",
    train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1,
)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, -train_scores.mean(axis=1), "o-", label="Train RMSE")
plt.plot(train_sizes, -val_scores.mean(axis=1), "o-", label="Val RMSE")
plt.xlabel("Training set size")
plt.ylabel("RMSE")
plt.legend()
plt.title(f"Learning curve - {best['name']}")
plt.show()

**Diagnosis:** _(TODO(you) - do train/val curves converge? Gap size? Under- or over-fitting? What would you do next?)_

## Milestone 6 - Final evaluation on the held-out test set (run exactly once)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

test_preds = best_model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, test_preds))
test_mae = mean_absolute_error(y_test, test_preds)
test_r2 = r2_score(y_test, test_preds)

print(f"Test RMSE: {test_rmse:.4f}")
print(f"Test MAE:  {test_mae:.4f}")
print(f"Test R^2:  {test_r2:.4f}")

In [ ]:
residuals = y_test.values - test_preds
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(test_preds, residuals, alpha=0.3, s=10)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Predicted ($100k)")
axes[0].set_ylabel("Residual")
axes[1].hist(residuals, bins=50)
axes[1].set_xlabel("Residual")
fig.tight_layout()
plt.show()

## Milestone 7 - Copy your final numbers into `reports/evaluation_report.md`

Do this now, while the numbers are in front of you - don't leave it for later.